# Stage 2: Categorical Variable Encoding
**Member:** M2 (Student ID: IT002)  
**Assigned Preprocessing Technique:** Encoding Categorical Variables (One-Hot & Binary Encoding)  
**Dataset:** [Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) (UCI Machine Learning Repository)  
**Pipeline Position:** Stage 2 (Sequential)  
**Input:** `results/outputs/stage1_missing_handled.csv`  
**Output:** `results/outputs/stage2_encoded.csv`

---

## 1. Explanation of the Technique

Machine learning models (e.g. Logistic Regression, Neural Networks, SVMs, and tree ensembles) operate on numerical matrices. Categorical variables expressed as arbitrary numeric identifiers must be converted into appropriate numerical representations:
- **Binary / Indicator Encoding**: For two-level variables (e.g. `SEX`), mapping to $0$ and $1$ creates a single feature without redundant degrees of freedom.
- **One-Hot Encoding (OHE)**: Creates $k$ binary columns for a categorical feature with $k$ levels. This ensures that no artificial numerical ordering or magnitude bias is imposed on nominal classes.
- **Ordinal Encoding**: Assigns monotonic integer ranks ($1 < 2 < \dots < k$) when classes possess a legitimate domain hierarchy.

---

## 2. Justification for THIS Dataset Specifically

In the Credit Card dataset, there are three demographic categorical attributes:
1. `SEX`: 1 = Male, 2 = Female.
   - Natural order: None.
   - Strategy: Convert to `SEX_FEMALE` ($1$ if Female, $0$ if Male). This avoids creating two perfectly collinear columns (`SEX_1` and `SEX_2`).
2. `MARRIAGE`: 1 = Married, 2 = Single, 3 = Others.
   - Natural order: None. Married vs Single vs Other does not follow an arithmetic progression.
   - Strategy: One-Hot Encoding (`MARRIAGE_1`, `MARRIAGE_2`, `MARRIAGE_3`).
3. `EDUCATION`: 1 = Graduate School, 2 = University, 3 = High School, 4 = Others.
   - While education has a loose societal hierarchy, Graduate School (1) is coded with a smaller integer than High School (3). If fed directly into a linear model, the model would falsely assume High School has 3x the magnitude of Graduate School!
   - Strategy: One-Hot Encoding (`EDUCATION_1`, `EDUCATION_2`, `EDUCATION_3`, `EDUCATION_4`) allows the model to estimate independent default probabilities for each educational tier without assuming linearity.

### Why this must be Stage 2:
Stage 2 depends directly on Stage 1 having cleansed the undocumented levels (`0, 5, 6` for Education, `0` for Marriage). If encoding were done before Stage 1, spurious columns would be formed. Encoding now ensures downstream outlier detection (M3) and feature scaling (M5) operate on properly structured numeric vectors.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure output directories exist
os.makedirs('results/outputs', exist_ok=True)
os.makedirs('results/eda_visualizations', exist_ok=True)

# 1. Load Stage 1 cleaned output
input_path = 'results/outputs/stage1_missing_handled.csv'
df_s1 = pd.read_csv(input_path)
print(f"Loaded Stage 1 Data: {df_s1.shape[0]} rows, {df_s1.shape[1]} columns")
df_s1[['SEX', 'EDUCATION', 'MARRIAGE']].head()


Loaded Stage 1 Data: 30000 rows, 25 columns


In [2]:
# 2. Perform Binary and One-Hot Encoding
df_encoded = df_s1.copy()

# Binary encoding for SEX: 1=Male -> 0, 2=Female -> 1
df_encoded['SEX_FEMALE'] = (df_encoded['SEX'] == 2).astype(int)

# One-hot encoding for EDUCATION (1: Grad School, 2: University, 3: High School, 4: Others)
for edu_code, edu_name in [(1, 'EDUCATION_1'), (2, 'EDUCATION_2'), (3, 'EDUCATION_3'), (4, 'EDUCATION_4')]:
    df_encoded[edu_name] = (df_encoded['EDUCATION'] == edu_code).astype(int)

# One-hot encoding for MARRIAGE (1: Married, 2: Single, 3: Others)
for marr_code, marr_name in [(1, 'MARRIAGE_1'), (2, 'MARRIAGE_2'), (3, 'MARRIAGE_3')]:
    df_encoded[marr_name] = (df_encoded['MARRIAGE'] == marr_code).astype(int)

# Drop original categorical columns
df_encoded.drop(columns=['SEX', 'EDUCATION', 'MARRIAGE'], inplace=True)

# Reorder columns: Put ID first, encoded demographics next, then repayment/financial features
new_cols = ['ID', 'SEX_FEMALE', 'EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4',
            'MARRIAGE_1', 'MARRIAGE_2', 'MARRIAGE_3'] + [c for c in df_encoded.columns if c not in [
            'ID', 'SEX_FEMALE', 'EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4',
            'MARRIAGE_1', 'MARRIAGE_2', 'MARRIAGE_3']]
df_encoded = df_encoded[new_cols]

print(f"Encoded Dataset Shape: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")
print("\nEncoded feature preview:")
df_encoded[['ID', 'SEX_FEMALE', 'EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4',
            'MARRIAGE_1', 'MARRIAGE_2', 'MARRIAGE_3']].head()


Encoded Dataset Shape: 30000 rows, 30 columns


In [3]:
# 3. Save Stage 2 output CSV
output_path = 'results/outputs/stage2_encoded.csv'
df_encoded.to_csv(output_path, index=False)
print(f"Successfully exported Stage 2 output to: {output_path}")


Successfully exported Stage 2 output to: results/outputs/stage2_encoded.csv


In [4]:
# 4. EDA Visualization: Default Rates Across Encoded Demographic Attributes
target = 'default payment next month'

edu_labels = ['Grad School (1)', 'University (2)', 'High School (3)', 'Others (4)']
edu_rates = [df_encoded[df_encoded[f'EDUCATION_{i}'] == 1][target].mean() * 100 for i in range(1, 5)]

marr_labels = ['Married (1)', 'Single (2)', 'Others (3)']
marr_rates = [df_encoded[df_encoded[f'MARRIAGE_{i}'] == 1][target].mean() * 100 for i in range(1, 4)]

sex_labels = ['Male (0)', 'Female (1)']
sex_rates = [df_encoded[df_encoded['SEX_FEMALE'] == s][target].mean() * 100 for s in [0, 1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plt.subplots_adjust(wspace=0.3)

# Education plot
axes[0].bar(edu_labels, edu_rates, color=['#3498db', '#2980b9', '#1abc9c', '#16a085'])
axes[0].set_title('Default Rate by Education Tier', fontweight='bold')
axes[0].set_ylabel('Default Rate (%)')
axes[0].set_ylim(0, 30)
axes[0].tick_params(axis='x', rotation=25)
for i, v in enumerate(edu_rates):
    axes[0].text(i, v + 0.8, f"{v:.1f}%", ha='center', fontweight='bold')

# Marriage plot
axes[1].bar(marr_labels, marr_rates, color=['#9b59b6', '#8e44ad', '#6c3483'])
axes[1].set_title('Default Rate by Marital Status', fontweight='bold')
axes[1].set_ylabel('Default Rate (%)')
axes[1].set_ylim(0, 30)
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(marr_rates):
    axes[1].text(i, v + 0.8, f"{v:.1f}%", ha='center', fontweight='bold')

# Sex plot
axes[2].bar(sex_labels, sex_rates, color=['#34495e', '#e74c3c'])
axes[2].set_title('Default Rate by Sex (SEX_FEMALE)', fontweight='bold')
axes[2].set_ylabel('Default Rate (%)')
axes[2].set_ylim(0, 30)
for i, v in enumerate(sex_rates):
    axes[2].text(i, v + 0.8, f"{v:.1f}%", ha='center', fontweight='bold')

plt.suptitle('M2: Default Risk Across Encoded Categorical Attributes', fontsize=14, fontweight='bold')
plot_path = 'results/eda_visualizations/m2_categorical_default_rates.png'
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"EDA plot saved to {plot_path}")


EDA plot saved to results/eda_visualizations/m2_categorical_default_rates.png


## 3. EDA Interpretation & Findings

1. **Education Level Impact**:
   - Graduate school clients exhibit the lowest default rate (**19.2%**), followed by University graduates (**23.7%**), and High School clients exhibiting the highest risk (**25.2%**). The "Others" tier has a default rate of **7.1%**.
   - This validates our decision to use One-Hot Encoding rather than linear rank encoding: default risk does not scale uniformly across levels.

2. **Marital Status Impact**:
   - Married cardholders show higher default rates (**23.5%**) compared to single individuals (**20.9%**).

3. **Sex Impact**:
   - Male cardholders (coded 0 in `SEX_FEMALE`) default at **24.2%**, whereas female cardholders (coded 1 in `SEX_FEMALE`) default at **20.8%**.

4. **Hand-off to Stage 3 (M3)**:
   - The categorical variables are fully encoded as numeric dummy features ($0$ and $1$).
   - Stage 3 can now proceed with outlier detection on the continuous financial features without interference from discrete categorical codes.
